Data Quality: Initial Missing-Value Assessment & Duplicate Analysis


### 1. Dataset Loading

**Objective:** Load the original dataset to perform Data Quality Assessment.

The dataset is read from the raw data directory using pandas. Since the dataset contains millions of records, it is loaded in chunks of 100,000 rows to manage the reading process. The chunks are then combined into a single DataFrame for subsequent analysis.

**Note:** Combining all chunks still requires enough RAM to store the complete dataset.
  

In [11]:

import pandas as pd

# Dataset path
DATA_PATH = "../data/raw/data.csv"

# Read the dataset in chunks of 100,000 rows
CHUNK_SIZE = 100_000

chunks = []

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    chunks.append(chunk)

# Combine all chunks into one DataFrame
df = pd.concat(chunks, ignore_index=True)

print("Dataset loaded successfully!")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")


Dataset loaded successfully!
Total rows: 8,714,356
Total columns: 20


2. Data Quality Assessment
2.1 Missing Values
We analyze missing values to assess data completeness and identify attributes that may require cleaning.

In [3]:
# Calculate the percentage of missing values for each attribute
missing_percentage = df.isnull().mean() * 100

# Display percentages rounded to two decimal places
missing_percentage.round(2)

ioCountry                       0.00
postid                          0.00
post_text                       0.00
application_name                0.00
post_language                   1.19
in_reply_to_postid             89.83
in_reply_to_accountid          89.96
post_time                       0.00
accountid                       0.00
account_profile_description    19.16
follower_count                  0.02
following_count                 0.01
account_creation_date           0.00
is_repost                       0.00
reposted_accountid             58.63
reposted_postid                58.63
hashtags                        0.00
urls                            0.00
account_mentions                0.00
is_control                      0.00
dtype: float64

Missing Values — Cleaning Decision

The initial analysis identified missing values in several attributes, particularly in_reply_to_postid, in_reply_to_accountid, reposted_postid, and reposted_accountid.

However, missing values will not be handled at this stage.

We will first investigate and handle duplicated records because removing duplicates may also eliminate some missing values and change the overall missing-value statistics.

In [4]:
# Count fully duplicated rows
df.duplicated().sum()

np.int64(1579)

In [5]:
# Count duplicated post IDs
df.duplicated(subset=["postid"]).sum()

np.int64(22909)

2.3 Investigation of Duplicated Post IDs
We identified 22,909 duplicated post IDs, compared to 1,565 fully duplicated rows.

We can safely delete the 1,565 duplicated rows

But for the 22,909, it means that some records share the same postid but have different values in other attributes.

To investigate this, we display examples of records with duplicated post IDs and compare their attributes.

In [27]:
# Create a working copy and remove fully duplicated records
df_clean = df.drop_duplicates().copy()

# Check the number of remaining records
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed duplicates:", len(df) - len(df_clean))

Original rows: 8714356
Cleaned rows: 8712777
Removed duplicates: 1579


In [7]:
# Inspect records sharing the same post ID
duplicate_posts = df[df.duplicated(subset=["postid"], keep=False)]

duplicate_posts.sort_values("postid").head(10)

,ioCountry,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,following_count,account_creation_date,is_repost,reposted_accountid,reposted_postid,hashtags,urls,account_mentions,is_control
4987290,russia_1,000185bbdd3796dea2a394b1fb30a106c8383d40d0,"Есть люди, которые родились на свет, чтобы идт...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,ru,NaN,NaN,2014-08-06 08:39:00,2fcf7636479946da380f01196e93de111e256fb54a,student,183,297,2014-05-06,False,NaN,NaN,[],[],[],False
8713685,russia_1,000185bbdd3796dea2a394b1fb30a106c8383d40d0,"Есть люди, которые родились на свет, чтобы идт...",62010408238d4a0f958ec97eec3db0a4f69bcd0d11,ru,NaN,NaN,2014-08-06 08:39:00,2fcf7636479946da380f01196e93de111e256fb54a,student,183,297,2014-05-06,False,NaN,NaN,[],[],[],False
2042411,iran_1,0003760bb300f3d26014c2f4d118e77b8b21bfb9e4,"#CARRIÓ CONTRA #UNASUR.Sobre #VENEZUELA, LA ap...",15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,es,NaN,NaN,2014-04-08 01:45:08,2a2fa4cb06cbea5871d09af8d5d70e194e6dace8b1,DESARROLLO WEB Y COMMUNITY MANAGER,613,927,2013-01-11,True,3848f3b40b7dbc65fa1f7b14a16951d7fbc563a651,d7778df81af602ac17bad28b1ddfc2fbf1f3433169,['CARRIÓ' 'UNASUR' 'VENEZUELA'],[],['3848f3b40b7dbc65fa1f7b14a16951d7fbc563a651'],True
4318079,russia_1,0003760bb300f3d26014c2f4d118e77b8b21bfb9e4,"#CARRIÓ CONTRA #UNASUR.Sobre #VENEZUELA, LA ap...",15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,es,NaN,NaN,2014-04-08 01:45:08,2a2fa4cb06cbea5871d09af8d5d70e194e6dace8b1,DESARROLLO WEB Y COMMUNITY MANAGER,612,926,2013-01-11,True,3848f3b40b7dbc65fa1f7b14a16951d7fbc563a651,d7778df81af602ac17bad28b1ddfc2fbf1f3433169,['CARRIÓ' 'UNASUR' 'VENEZUELA'],[],['3848f3b40b7dbc65fa1f7b14a16951d7fbc563a651'],True
8140339,uae,0005742c049d43d6ea736e8bbdfa381f7ef960a26a,https://46d0b09ddd39228b255979ec3111aac1660cf8...,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,und,NaN,NaN,2016-09-19 19:17:00,108d969e0b44b92a68c09aa26785238e39def38f25,‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏ابو سمبل،توماس،نحن الصامتين\n...,1539,170,2015-11-06,False,NaN,NaN,[],['https://46d0b09ddd39228b255979ec3111aac1660c...,[],False
8714251,uae,0005742c049d43d6ea736e8bbdfa381f7ef960a26a,https://46d0b09ddd39228b255979ec3111aac1660cf8...,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,und,NaN,NaN,2016-09-19 19:17:00,108d969e0b44b92a68c09aa26785238e39def38f25,‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏‏ابو سمبل،توماس،نحن الصامتين\n...,1539,170,2015-11-06,False,NaN,NaN,[],['https://46d0b09ddd39228b255979ec3111aac1660c...,[],False
7457402,uae,000703ce87672809b7545d752f4ee2833450ef97dc,الوضع الاقتصادي محرج، يجب العلاج الدقيق له، ول...,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,ar,NaN,NaN,2018-11-13 15:53:00,95e585b562f90955d190a1355690ff45f45e545209,لبنان أرض تحتوي كل شي ونحنا عنا كل شي Everyth...,46,137,2018-08-25,True,b3964e165abddd2a9ac6f192530572f598e510bbfe,2f4dde35510009de6466d0dff8c162c55d8a8917ac,[],[],['b3964e165abddd2a9ac6f192530572f598e510bbfe'],False
8714118,uae,000703ce87672809b7545d752f4ee2833450ef97dc,الوضع الاقتصادي محرج، يجب العلاج الدقيق له، ول...,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,ar,NaN,NaN,2018-11-13 15:53:00,95e585b562f90955d190a1355690ff45f45e545209,لبنان أرض تحتوي كل شي ونحنا عنا كل شي Everyth...,46,137,2018-08-25,True,b3964e165abddd2a9ac6f192530572f598e510bbfe,2f4dde35510009de6466d0dff8c162c55d8a8917ac,[],[],['b3964e165abddd2a9ac6f192530572f598e510bbfe'],False
3210744,iran_4,00086574bc5607acf77c9ba56f6ad4a782d0d2cc96,No Pride Just Prejudice #RepublicanAMovie,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,en,NaN,NaN,2017-10-17 21:40:10,e7f8b9b1148bf68cc2aa6b37a522f24dbf0fc2bba5,"⚓ Silicon Progressive, Health Innovator, onlin...",1334,2529,2013-03-24,True,31191cef20dda98e51e4fa607a7fca48cd0280b5d9,df4e50c6e9b63986ea34f9b1749c2ec2dc636bfc3b,['RepublicanAMovie'],[],['31191cef20dda98e51e4fa607a7fca48cd0280b5d9'],True
6838057,russia_1,00086574bc5607acf77c9ba56f6ad4a782d0d2cc96,No Pride Just Prejudice #RepublicanAMovie,15592fcc82d5064a8f4a3b89774e6cd0bbd2118aa0,en,N

Analysis of Conflicting Duplicate Records

We investigate records sharing the same postid to identify which attributes contain conflicting values.


In [8]:
# Find records sharing the same post ID in the cleaned dataset
duplicate_posts = df_clean[
    df_clean.duplicated(subset=["postid"], keep=False)
]

In [9]:
# Count how many post IDs have conflicting values in each attribute
conflicting_columns = (
    duplicate_posts
    .groupby("postid")
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

conflicting_columns

ioCountry                      20838
follower_count                 11228
following_count                 9929
account_profile_description      409
accountid                        234
account_creation_date            194
post_text                        148
urls                             117
hashtags                          75
reposted_postid                   67
account_mentions                  56
reposted_accountid                48
application_name                  39
post_language                     30
is_repost                         12
in_reply_to_accountid              4
is_control                         4
in_reply_to_postid                 2
post_time                          1
dtype: int64

### Duplicate Records: Important Clarifications

Anonymized identifiers: User IDs, post IDs, and URLs have been anonymized using hash functions. The anonymization may differ between ioCountry groups. Therefore, identifiers are not necessarily comparable across countries.

Conflicts within the same ioCountry: If records have the same postid and ioCountry but different accountid values, this indicates a potential data inconsistency that requires investigation

Other conflicting attributes: Differences in attributes such as post_text, post_time, and is_control should also be investigated to determine whether they represent actual inconsistencies.


In [10]:
# Select only the necessary columns from repeated posts
subset = duplicate_posts[
    ["ioCountry", "postid", "accountid"]
]

# Check conflicting account IDs within the same country and post
account_conflicts = (
    subset.groupby(["ioCountry", "postid"])["accountid"]
    .nunique(dropna=True)
)

# Keep groups with more than one distinct account ID
account_conflicts = account_conflicts[account_conflicts > 1]

print("Number of conflicting groups:", len(account_conflicts))

display(account_conflicts.head(10))

Number of conflicting groups: 226


ioCountry  postid                                    
ecuador    0d0e7f0542efc16cbc65c2f61d6808611d65406b1d    2
           17f61e99a642c36c4d46810638c7545e54816e5c9f    2
           1c6cc92180b8586a93efd3cf5280ba1cd374d78426    2
           264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa    2
           28b6cba67fa61104d09589ae6f0f15596063aaea76    2
           2b3c0f9ac6ba7304e575b6786e7f9689d1c04e452c    2
           330a30ec691b83588ce22264dcfb26a6a99c232f91    2
           4262f7363a33ef07b7f47e26da8a7d940cb5d10c25    2
           4d0634e79beb014248b5e8e711f078590b74edec51    2
           586572f6303500701b0157b4e384670f0fdbbb86b3    2
Name: accountid, dtype: int64

### What should we do now?
Before deleting anything, we need to understand how the records differ.

In [11]:
# Select the first 5 conflicting country-post pairs
example_pairs = account_conflicts.head(5).index

# Retrieve the corresponding records
examples = (
    duplicate_posts
    .set_index(["ioCountry", "postid"])
    .loc[example_pairs, [
        "accountid",
        "post_text",
        "post_time",
        "is_control"
    ]]
    .reset_index()
)

display(examples)

,ioCountry,postid,accountid,post_text,post_time,is_control
0,ecuador,0d0e7f0542efc16cbc65c2f61d6808611d65406b1d,fa3fe8929f0583d879d124ed805a103368afe1f43c,En menos de 2 meses Cedatos recibió $807.360 d...,2017-03-21 21:31:45,True
1,ecuador,0d0e7f0542efc16cbc65c2f61d6808611d65406b1d,a5cad6152d2da18181b8eeccaf7c3c94c22616f3f1,En menos de 2 meses Cedatos recibió $807.360 d...,2017-03-21 21:31:45,True
2,ecuador,17f61e99a642c36c4d46810638c7545e54816e5c9f,b11edd9b878259a855e0cecfcb532cef711937bfda,-264\nhttps://8b31e2b3f288f9a4b3496afe1d66d660...,2017-08-07 23:17:14,True
3,ecuador,17f61e99a642c36c4d46810638c7545e54816e5c9f,e21bf8f8d1e133575c1c3e25dc1d9d065f6684840d,-264\nhttps://ed4145c39cc1b31864b6a64d20835036...,2017-08-07 23:17:14,True
4,ecuador,1c6cc92180b8586a93efd3cf5280ba1cd374d78426,5cbcef399e68fc4a5ca6c9d0b43f4a21607addf449,-234\n#LaloBritoTrendy #DannaPaola #SofiaReyes...,2017-08-07 23:07:49,True
5,ecuador,1c6cc92180b8586a93efd3cf5280ba1cd374d78426,bc73d544af74f0905a1629c5ba1e72e9713ce83b39,-234\n#LaloBritoTrendy #DannaPaola #SofiaReyes...,2017-08-07 23:07:49,True
6,ecuador,264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa,2a36ad6794513dd37723f6fa93416ef351a63c39ea,Gracias a los ciudadanos de #Cotopaxi que nos ...,2018-07-09 23:58:54,True
7,ecuador,264bb4b4042cd5f46e5b9826b9d91f2ff4cba80bfa,81a2f99d3cfb9a5e8aed22c9db15833b1cd4c63bae,En mi callada tarde\nsurgió la lluvia incesant...,2018-07-09 23:58:54,True
8,ecuador,28b6cba67fa61104d09589ae6f0f15596063aaea76,bc73d544af74f0905a1629c5ba1e72e9713ce83b39,-237\n#LaloBritoTrendy #DannaPaola #SofiaReyes...,2017-08-07 23:08:19,True
9,ecuador,28b6cba67fa61104d09589ae6f0f15596063aaea76,ed6510731c58eb8461183197325b7bc51eccdb59fa,-237\n#LaloBritoTrendy #DannaPaola #SofiaReyes...,2017-08-07 23:08:19,True


In [12]:
# Retrieve all records belonging to the 226 conflicting groups
conflict_keys = account_conflicts.index.to_frame(index=False)

conflict_records = duplicate_posts.merge(
    conflict_keys,
    on=["ioCountry", "postid"],
    how="inner"
)

# Compare other important attributes within these groups
# Compare all attributes within the 226 conflicting groups
all_conflicts = (
    conflict_records
    .groupby(["ioCountry", "postid"])
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

# Display only attributes with differences
all_conflicts = all_conflicts[all_conflicts > 0]

print("Total conflicting groups:", len(account_conflicts))

display(all_conflicts)


Total conflicting groups: 226


accountid                      226
follower_count                 200
account_creation_date          193
following_count                186
account_profile_description    146
post_text                       77
urls                            61
hashtags                        42
application_name                38
account_mentions                22
post_language                   20
reposted_accountid              17
reposted_postid                 17
is_repost                       12
in_reply_to_accountid            3
in_reply_to_postid               2
post_time                        1
is_control                       1
dtype: int64

### Investigation of Conflicting Records

We examined all attributes within the **226 groups with conflicting account IDs** to better understand the nature of these inconsistencies.

**Main findings:**
- 226 groups have different `accountid` values.
- 193 groups have different `account_creation_date` values.
- 186 groups have different `following_count` values.
- 77 groups have different `post_text` values.
- Only 1 group differs in `post_time`, and 1 in `is_control`.

The conflicts involving `post_time` and `is_control` belong to the same group (`russia_1`), where two records have different account IDs, different post texts, publication times six seconds apart, and opposite IO/control labels.

**Interpretation:**

The differences in account-related attributes, particularly account creation dates, suggest that some records may represent different accounts.

Differences in post content and repost information may indicate distinct posts sharing an anonymized identifier or other data inconsistencies.

**Conclusion:**

Since our main objective is user profiling, the most important concern is correctly associating posts with their accounts.

In [13]:
# Identify rows belonging to the 226 conflicting groups
duplicate_keys = pd.MultiIndex.from_frame(
    duplicate_posts[["ioCountry", "postid"]]
)

affected_mask = duplicate_keys.isin(account_conflicts.index)

affected_indices = duplicate_posts.index[affected_mask]

print("Conflicting groups:", len(account_conflicts))
print("Affected rows:", len(affected_indices))

print(
    "Percentage of dataset:",
    round(len(affected_indices) / len(df_clean) * 100, 4),
    "%"
)

Conflicting groups: 226
Affected rows: 460
Percentage of dataset: 0.0053 %


### Impact of Conflicting Account IDs and Proposed Strategy

**Results:**
- Conflicting `(ioCountry, postid)` groups: **226**
- Total affected records: **460**
- Percentage of cleaned dataset: **0.0053%**

These groups contain records with ambiguous post identifiers or account associations.

**Proposed investigation:**

Before removing these records, we will investigate whether the posting history and other attributes of the associated accounts provide evidence that helps explain the conflicts.

Topic similarity alone will not be considered sufficient to determine post ownership, particularly because coordinated accounts may publish similar content.

We will try checking to see if it helps

In [14]:
# Select the first conflicting group
example = conflict_records.iloc[0]

country = example["ioCountry"]
post_id = example["postid"]

# Get the conflicting records
pair = conflict_records[
    (conflict_records["ioCountry"] == country) &
    (conflict_records["postid"] == post_id)
]

display(pair[
    ["ioCountry", "postid", "accountid", "post_text"]
])

# Identify the accounts involved
accounts = pair["accountid"].dropna().unique()

# Examine other posts from those accounts
for account in accounts:
    account_posts = df_clean.loc[
        (df_clean["ioCountry"] == country) &
        (df_clean["accountid"] == account) &
        (df_clean["postid"] != post_id),
        ["post_time", "post_text", "post_language"]
    ]

    print(f"\nAccount: {account}")
    print("Other posts found:", len(account_posts))

    display(account_posts.head(10))

,ioCountry,postid,accountid,post_text
0,ecuador,c62bfbf48df48628bf1764a380691ca74ac6a858fe,b11edd9b878259a855e0cecfcb532cef711937bfda,4943\n#LaloBritoTrendy #DannaPaola #LosRules #...
1,ecuador,c62bfbf48df48628bf1764a380691ca74ac6a858fe,9743f71178cf2ed5e9ff9100700f6f6307839a2560,4943\n#LaloBritoTrendy #DannaPaola #LosRules #...



Account: b11edd9b878259a855e0cecfcb532cef711937bfda
Other posts found: 197


,post_time,post_text,post_language
60647,2017-07-28 22:57:31,4930\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60668,2017-07-28 22:57:35,4931\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60690,2017-07-28 22:57:39,4932\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60702,2017-07-28 22:57:43,4933\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60731,2017-07-28 22:57:47,4934\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60750,2017-07-28 22:57:51,4935\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60766,2017-07-28 22:57:57,4936\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60789,2017-07-28 22:58:08,4937\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60815,2017-07-28 22:58:22,yey :p\n#LaloBritoTrendy #DannaPaola #LosRules...,en
60816,2017-07-28 22:58:47,4938\n#LaloBritoTrendy #DannaPaola #LosRules #...,und



Account: 9743f71178cf2ed5e9ff9100700f6f6307839a2560
Other posts found: 98


,post_time,post_text,post_language
60644,2017-07-28 22:57:27,4929\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60655,2017-07-28 22:57:31,4930\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60676,2017-07-28 22:57:35,4931\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60700,2017-07-28 22:57:39,4932\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60710,2017-07-28 22:57:43,4933\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60729,2017-07-28 22:57:47,4934\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60757,2017-07-28 22:57:51,4935\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60763,2017-07-28 22:57:57,4936\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60784,2017-07-28 22:58:08,4937\n#LaloBritoTrendy #DannaPaola #LosRules #...,und
60814,2017-07-28 22:58:22,yey :p\n#LaloBritoTrendy #DannaPaola #LosRules...,en


### Investigation of Account Posting Histories

To investigate conflicting account IDs, we examined the posting histories of two accounts associated with the same `(ioCountry, postid)`.

**Observations:**
- Both accounts published posts containing similar numbered messages and hashtags.
- Several posts have matching publication timestamps.
- Their posting behavior appears highly similar.

This suggests a possible systematic relationship between the accounts. However, the similarity alone is insufficient to determine whether the records represent coordinated activity, duplicated account information, or identifier inconsistencies.

maybe execlude

### Analysis of Remaining Repeated Post IDs

After identifying the groups with conflicting account IDs, we continue investigating repeated `(ioCountry, postid)` combinations.

The objective is to determine whether additional repeated records could affect user-level features, particularly post counts and posting frequency.


In [15]:
# Count records for each country-post combination
pair_counts = (
    duplicate_posts
    .groupby(["ioCountry", "postid"], dropna=False)
    .size()
)

# Keep combinations appearing more than once
repeated_pairs = pair_counts[pair_counts > 1]

# Separate the 226 groups with conflicting account IDs
remaining_pairs = repeated_pairs.drop(
    account_conflicts.index,
    errors="ignore"
)

print("Total repeated (ioCountry, postid) groups:", len(repeated_pairs))

print("Groups with conflicting account IDs:", len(account_conflicts))

print("Other repeated groups:", len(remaining_pairs))

print("Extra rows in other repeated groups:",
      int((remaining_pairs - 1).sum()))

Total repeated (ioCountry, postid) groups: 296
Groups with conflicting account IDs: 226
Other repeated groups: 70
Extra rows in other repeated groups: 71


### Analysis of Conflicting Attributes in Remaining Repeated Posts

We identified 84 repeated `(ioCountry, postid)` groups without conflicting known account IDs, containing 169 records.

Before deciding how to handle these repetitions, we will examine all attributes to identify which values differ within each group.


In [16]:
# Retrieve records belonging to the 84 remaining groups
remaining_mask = pd.MultiIndex.from_frame(
    duplicate_posts[["ioCountry", "postid"]]
).isin(remaining_pairs.index)

remaining_records = duplicate_posts.loc[remaining_mask].copy()

# Count groups with different values for each attribute
remaining_conflicts = (
    remaining_records
    .groupby(["ioCountry", "postid"], dropna=False)
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

# Show only attributes containing differences
remaining_conflicts = remaining_conflicts[
    remaining_conflicts > 0
]

print("Total groups:", len(remaining_pairs))
print("Total records:", len(remaining_records))

display(remaining_conflicts)

Total groups: 70
Total records: 141


post_text             70
urls                  54
reposted_postid       50
account_mentions      34
hashtags              32
reposted_accountid    31
post_language          9
is_control             1
follower_count         1
following_count        1
dtype: int64

### Results of the Remaining Duplicate Analysis

We examined the 84 repeated `(ioCountry, postid)` groups, containing 169 records.
No differences were found in `accountid` or `post_time` within these groups.

**Interpretation:**

Although the account IDs are consistent within each group, many records differ in content-related attributes. Therefore, we cannot assume that every repeated `(ioCountry, postid)` combination represents an identical observation.

The differences in `is_control` are especially relevant because this attribute distinguishes information-operation posts from control posts.

**Next step:** Investigate the 15 groups with conflicting `is_control` values before deciding how to handle the remaining repeated records.

In [17]:
# Find groups with different is_control values
label_conflicts = (
    remaining_records
    .groupby(["ioCountry", "postid"], dropna=False)["is_control"]
    .nunique(dropna=False)
)

label_conflicts = label_conflicts[label_conflicts > 1]

# Retrieve the corresponding records
label_mask = pd.MultiIndex.from_frame(
    remaining_records[["ioCountry", "postid"]]
).isin(label_conflicts.index)

label_records = remaining_records.loc[
    label_mask,
    ["ioCountry", "postid", "accountid",
     "post_text", "post_time", "is_control"]
].copy()

# Check whether differences involve different data types
label_records["label_type"] = (
    label_records["is_control"]
    .map(lambda x: type(x).__name__)
)

print("Groups with conflicting labels:", len(label_conflicts))

display(label_records.head(20))

Groups with conflicting labels: 1


,ioCountry,postid,accountid,post_text,post_time,is_control,label_type
1039190,ecuador,1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa,1decb332cd4b7c2dfa969ebc3f5e9e604a5a183f59,¡Nuestras Islas Galápagos 🌊🐠🐢 son un lugar úni...,2019-03-10 15:59:00,True,str
1039193,ecuador,1d6e232d8247d28bd95b1fc6df75cab64ba3cf2afa,1decb332cd4b7c2dfa969ebc3f5e9e604a5a183f59,¡Nuestras Islas Galápagos 🌊🐠🐢 son un lugar úni...,2019-03-10 15:59:00,False,str


### Investigation of IO/Control Label Conflicts

Among the 84 remaining repeated `(ioCountry, postid)` groups, we identified **15 groups with conflicting `is_control` values**.

These conflicts involve actual `True` and `False` values, rather than missing values or different data representations.

**Decision:** We will document these conflicts and check if those records have the same or different post_text


In [18]:
# Compare post texts within the 15 groups
label_text_check = (
    label_records
    .groupby(["ioCountry", "postid"])["post_text"]
    .nunique(dropna=False)
)

print("Groups with identical text:",
      (label_text_check == 1).sum())

print("Groups with different text:",
      (label_text_check > 1).sum())

Groups with identical text: 0
Groups with different text: 1


### Investigation of IO/Control Label Conflicts

Among the 84 remaining repeated `(ioCountry, postid)` groups, we identified **15 groups with conflicting `is_control` values** (`True` and `False`).

Further investigation showed that:
- **14 groups** have identical `post_text` values despite different IO/control labels.
- **1 group** has different `post_text` values and conflicting IO/control labels.

These results suggest potential classification inconsistencies, particularly in the 14 groups with identical post content.

**Decision:** We will document these inconsistencies and postpone their resolution until we establish a justified cleaning strategy, potentially following instructor clarification. 

Now we will investigate the other 69 groups

In [19]:
# Exclude the 15 groups with conflicting is_control
keys = pd.MultiIndex.from_frame(
    remaining_records[["ioCountry", "postid"]]
)

other_records = remaining_records.loc[
    ~keys.isin(label_conflicts.index)
].copy()

# Check which attributes differ
other_conflicts = (
    other_records
    .groupby(["ioCountry", "postid"])
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

print("Remaining groups:",
      other_records.groupby(["ioCountry", "postid"]).ngroups)

display(other_conflicts[other_conflicts > 0])

# Verify missing values in post_text
print("\nMissing post_text values:",
      other_records["post_text"].isna().sum())

# Check groups with genuinely different non-missing texts
text_conflicts = (
    other_records
    .groupby(["ioCountry", "postid"])["post_text"]
    .nunique(dropna=True)
)

print("Groups with different non-missing post texts:",
      (text_conflicts > 1).sum())

Remaining groups: 69


post_text             69
urls                  53
reposted_postid       50
account_mentions      34
hashtags              32
reposted_accountid    31
post_language          9
dtype: int64


Missing post_text values: 0
Groups with different non-missing post texts: 69


### Possible Explanation for Different Post Texts

The 69 repeated groups have identical `ioCountry`, `postid`, `accountid`, `post_time`, and `is_control` values but different post texts.

One possible explanation is that posts were edited after publication, while their original publication timestamps remained unchanged.

Additionally, If the post was edited, the differences in hashtags and URLs could be explained by the change in content.

### Duplicate Records Handling — Pending Decisions

The duplicate analysis identified 310 repeated `(ioCountry, postid)` groups after removing fully identical records.

These groups were classified into:
- **226 groups:** Conflicting account identifiers.
- **15 groups:** Conflicting IO/control labels without multiple known account identifiers.
- **69 groups:** Consistent account identifiers and labels but different content-related attributes.

**Handling status:**

Fully identical records have already been removed.

The remaining duplicate cases have been investigated and documented, but no further cleaning has been applied because the correct handling strategy cannot be determined reliably from the available information.

**Important:** The unresolved records must be addressed before constructing the final user profiles, since they could affect post counts and other behavioral features.

maybe execlude all

In [20]:
# Examine the data types of all attributes
display(df_clean.dtypes)

ioCountry                       str
postid                          str
post_text                       str
application_name                str
post_language                   str
in_reply_to_postid              str
in_reply_to_accountid           str
post_time                       str
accountid                       str
account_profile_description     str
follower_count                  str
following_count                 str
account_creation_date           str
is_repost                      bool
reposted_accountid              str
reposted_postid                 str
hashtags                        str
urls                            str
account_mentions                str
is_control                      str
dtype: object

In [21]:
# Inspect numeric validity without modifying the data
for col in ["follower_count", "following_count"]:

    original = df_clean[col]
    numeric = pd.to_numeric(original, errors="coerce")

    invalid = original.notna() & numeric.isna()

    print(f"\nAttribute: {col}")
    print("Invalid numeric values:", invalid.sum())
    print("Negative values:", (numeric < 0).sum())

    print("Examples of invalid values:")
    display(original[invalid].value_counts().head(10))


Attribute: follower_count
Invalid numeric values: 2265
Negative values: 977
Examples of invalid values:


follower_count
error      281
unknown    268
NaN-ish    258
none       256
many       236
4O00       210
12abc      202
1,2,3      193
??42       191
ten        170
Name: count, dtype: int64


Attribute: following_count
Invalid numeric values: 2160
Negative values: 1029
Examples of invalid values:


following_count
NaN-ish    258
unknown    249
many       244
none       243
error      227
12abc      212
ten        200
??42       193
1,2,3      167
4O00       167
Name: count, dtype: int64

### Numeric Attribute Validation
**Proposed handling:**
- The first point means converting values that represent valid numbers into an actual numeric data type
- Replace genuinely invalid and negative values with `NaN`, rather than zero, then solve them when handling missing values.


In [22]:
for col in ["follower_count", "following_count"]:

    numeric = pd.to_numeric(df_clean[col], errors="coerce")

    invalid_values = df_clean.loc[
        df_clean[col].notna() & numeric.isna(), col
    ]

    print(f"\n--- {col} ---")

    # Inspect suspicious values containing digits
    suspicious = invalid_values[
        invalid_values.astype(str).str.contains(r"\d")
    ].drop_duplicates().head(15)

    print("Invalid values containing digits:")
    for value in suspicious:
        print(repr(value))

    # Inspect negative values
    print("\nExamples of negative values:")
    display(numeric[numeric < 0].value_counts().head(10))


--- follower_count ---
Invalid values containing digits:
'??42'
'1,2,3'
'12abc'
'4O00'

Examples of negative values:


follower_count
-292406.0    1
-968071.0    1
-703165.0    1
-121883.0    1
-597799.0    1
-379867.0    1
-19464.0     1
-963995.0    1
-311107.0    1
-75246.0     1
Name: count, dtype: int64


--- following_count ---
Invalid values containing digits:
'1,2,3'
'12abc'
'4O00'
'??42'

Examples of negative values:


following_count
-232627.0    1
-308326.0    1
-968542.0    1
-46059.0     1
-47596.0     1
-142242.0    1
-449407.0    1
-688408.0    1
-740065.0    1
-737830.0    1
Name: count, dtype: int64

### Cleaning Numeric Attributes

We will clean follower_count and following_count by converting valid numerical strings to numeric values and replacing non-convertible and negative values with NaN.

In [23]:
# Clean follower_count and following_count

for col in ["follower_count", "following_count"]:

    # Convert to numeric; invalid values become NaN
    df_clean[col] = pd.to_numeric(
        df_clean[col], errors="coerce"
    )

    # Replace negative values with NaN
    df_clean.loc[df_clean[col] < 0, col] = float("nan")

    # Verify the results
    print(f"\n{col}:")
    print("Data type:", df_clean[col].dtype)
    print("Negative values remaining:", (df_clean[col] < 0).sum())
    print("Total missing values:", df_clean[col].isna().sum())


follower_count:
Data type: float64
Negative values remaining: 0
Total missing values: 4560

following_count:
Data type: float64
Negative values remaining: 0
Total missing values: 4381


Now lets check the post_time and account_creation_date

In [24]:
# Inspect date attributes before conversion
date_columns = ["post_time", "account_creation_date"]

for col in date_columns:
    print(f"\n--- {col} ---")
    print("Current data type:", df_clean[col].dtype)
    print("Missing values:", df_clean[col].isna().sum())

    print("Example values:")
    display(df_clean[col].dropna().head(10))


--- post_time ---
Current data type: str
Missing values: 0
Example values:


0    2010-03-13 02:52:00
1    2010-03-13 03:08:00
2    2010-03-13 15:26:00
3    2010-03-13 15:32:00
4    2010-03-13 15:35:00
5    2010-03-13 15:38:00
6    2010-04-13 00:19:00
7    2010-04-13 00:36:00
8    2010-04-23 14:51:00
9    2010-04-23 14:53:00
Name: post_time, dtype: str


--- account_creation_date ---
Current data type: str
Missing values: 0
Example values:


0    2010-03-11
1    2010-03-11
2    2010-01-28
3    2010-01-28
4    2010-01-28
5    2010-01-28
6    2009-06-19
7    2009-06-19
8    2010-03-11
9    2010-03-11
Name: account_creation_date, dtype: str

Both post_time and account_creation_date should be converted converted from strings into datetime formats.

Assign UTC to post_time without changing its original clock time.

Convert invalid or unparseable dates to NaT.

Address any resulting missing values during the Missing Values Analysis.

In [25]:
# Convert post_time to datetime (UTC)
df_clean["post_time"] = pd.to_datetime(
    df_clean["post_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
    utc=True
)

# Convert account_creation_date to datetime
df_clean["account_creation_date"] = pd.to_datetime(
    df_clean["account_creation_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Verify the results
for col in ["post_time", "account_creation_date"]:
    print(f"\n--- {col} ---")
    print("Data type:", df_clean[col].dtype)
    print("Invalid/missing dates:", df_clean[col].isna().sum())
    display(df_clean[col].head(5))


--- post_time ---
Data type: datetime64[us, UTC]
Invalid/missing dates: 0


0   2010-03-13 02:52:00+00:00
1   2010-03-13 03:08:00+00:00
2   2010-03-13 15:26:00+00:00
3   2010-03-13 15:32:00+00:00
4   2010-03-13 15:35:00+00:00
Name: post_time, dtype: datetime64[us, UTC]


--- account_creation_date ---
Data type: datetime64[us]
Invalid/missing dates: 0


0   2010-03-11
1   2010-03-11
2   2010-01-28
3   2010-01-28
4   2010-01-28
Name: account_creation_date, dtype: datetime64[us]

Check logical date inconsistencies

An account cannot normally publish posts before its creation date.

In [26]:
# Compare calendar dates (without timezone)
post_dates = (
    df_clean["post_time"]
    .dt.tz_localize(None)
    .dt.normalize()
)

# Find posts published before account creation
invalid_dates = (
    post_dates < df_clean["account_creation_date"]
)

print("Posts before account creation:",
      invalid_dates.sum())

print("Percentage of dataset:",
      round(invalid_dates.mean() * 100, 4), "%")

# Display examples
display(
    df_clean.loc[
        invalid_dates,
        ["ioCountry", "accountid", "postid",
         "post_time", "account_creation_date"]
    ].head(10)
)

Posts before account creation: 0
Percentage of dataset: 0.0 %


,ioCountry,accountid,postid,post_time,account_creation_date


Validate is_control and is_repost

These are supposed to be two Boolean attributes in our dataset.

In [27]:
# Inspect Boolean attributes
for col in ["is_control", "is_repost"]:
    print(f"\n--- {col} ---")
    print("Data type:", df_clean[col].dtype)

    # Count each value, including missing values
    display(df_clean[col].value_counts(dropna=False))

    # Show Python types
    print("Python types:")
    display(df_clean[col].map(type).value_counts())


--- is_control ---
Data type: str


is_control
False          5229135
True           3482530
is_control?       1112
Name: count, dtype: int64

Python types:


is_control
<class 'str'>    8712777
Name: count, dtype: int64


--- is_repost ---
Data type: bool


is_repost
False    5108257
True     3604520
Name: count, dtype: int64

Python types:


is_repost
<class 'bool'>    8712777
Name: count, dtype: int64

The column of is_control contains mixed types: strings and Boolean values.

There are 1,112 invalid values ("is_control?") that cannot reliably be classified as IO or control.

We'll convert the string representations to actual Booleans and replace the invalid label with a missing value.

In [28]:
# Standardize is_control to Boolean values
df_clean["is_control"] = (
    df_clean["is_control"]
    .replace({
        "True": True,
        "False": False,
        "is_control?": pd.NA
    })
    .astype("boolean")
)

# Verify the results
print("Data type:", df_clean["is_control"].dtype)
print("Missing values:", df_clean["is_control"].isna().sum())

display(df_clean["is_control"].value_counts(dropna=False))

Data type: boolean
Missing values: 1112


is_control
False    5229135
True     3482530
<NA>        1112
Name: count, dtype: Int64

Next - Check repost-related inconsistencies

Now let's investigate whether is_repost agrees with the repost identifiers.

like if is_repost is True, the reposted_postid should have a value

and if is_repost is False, the reposted_postid should should not have a value

In [29]:
# Check consistency between repost flag and repost identifiers

for col in ["reposted_postid", "reposted_accountid"]:

    has_value = (
        df_clean[col].notna() &
        df_clean[col].ne("")
    )

    print(f"\n--- {col} ---")

    print("Reposts with identifier:",
          (df_clean["is_repost"] & has_value).sum())

    print("Reposts without identifier:",
          (df_clean["is_repost"] & ~has_value).sum())

    print("Non-reposts with identifier:",
          (~df_clean["is_repost"] & has_value).sum())

    print("Non-reposts without identifier:",
          (~df_clean["is_repost"] & ~has_value).sum())


--- reposted_postid ---
Reposts with identifier: 3604520
Reposts without identifier: 0
Non-reposts with identifier: 0
Non-reposts without identifier: 5108257

--- reposted_accountid ---
Reposts with identifier: 3604520
Reposts without identifier: 0
Non-reposts with identifier: 0
Non-reposts without identifier: 5108257


NO inconsistencies

now we will move to checking inconsistencies between in_reply_to_postid and in_reply_to_accountid

Unlike reposts, our dataset doesn't have an is_reply Boolean column. Therefore, we'll check whether the two reply identifiers are consistently provided.

In [30]:
# Check consistency between reply identifiers
has_reply_post = (
    df_clean["in_reply_to_postid"].notna() &
    df_clean["in_reply_to_postid"].ne("")
)

has_reply_account = (
    df_clean["in_reply_to_accountid"].notna() &
    df_clean["in_reply_to_accountid"].ne("")
)

print("Both identifiers present:",
      (has_reply_post & has_reply_account).sum())

print("Only reply post ID present:",
      (has_reply_post & ~has_reply_account).sum())

print("Only reply account ID present:",
      (~has_reply_post & has_reply_account).sum())

print("Neither identifier present:",
      (~has_reply_post & ~has_reply_account).sum())

Both identifiers present: 811470
Only reply post ID present: 74608
Only reply account ID present: 63456
Neither identifier present: 7763243


Reply Attribute Consistency — Results
Interpretation:

Possible explanations include incomplete information in the original data source, unavailable reply metadata, or limitations during data collection or processing.

However, the dataset does not provide sufficient information to determine the exact cause.

Cleaning decision:

We will preserve these records without modification because missing identifiers cannot be reliably reconstructed.

### Categorical Attribute Validation

We examine `ioCountry`, `application_name`, and `post_language` to identify potential errors and inconsistencies by checking their values

ioCountry: Typos, inconsistent capitalization, extra spaces, unexpected country-operation labels

post_language: Invalid language labels, duplicate representations such as EN and en, unexpected values

In [31]:
# Inspect categorical attributes
categorical_cols = [
    "ioCountry",
    "post_language"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")

    print("Unique categories:", df_clean[col].nunique())
    print("Missing values:", df_clean[col].isna().sum())

    # Check empty and whitespace-only strings
    empty_count = (
        df_clean[col]
        .dropna()
        .str.strip()
        .eq("")
        .sum()
    )

    print("Empty or whitespace-only values:", empty_count)

    print("\nTop 15 most frequent categories:")
    display(
        df_clean[col]
        .value_counts(dropna=False)
        .head(15)
    )


--- ioCountry ---
Unique categories: 5
Missing values: 0
Empty or whitespace-only values: 0

Top 15 most frequent categories:


ioCountry
russia_1    3650082
uae         1707792
ecuador     1470529
iran_1      1322789
iran_4       561585
Name: count, dtype: int64


--- post_language ---
Unique categories: 70
Missing values: 103727
Empty or whitespace-only values: 0

Top 15 most frequent categories:


post_language
en     2497602
ru     1568927
ar     1378222
es     1360396
fr      419150
und     373032
fa      317209
pt      134498
tr      131028
NaN     103727
ja       66801
in       58862
it       48181
uk       40395
de       38424
Name: count, dtype: int64

We examined ioCountry, and post_language.

Missing post_language values will be handled during the Missing Values Analysis.

In [32]:
# Display all language categories and their frequencies
display(
    df_clean["post_language"]
    .value_counts(dropna=False)
)

post_language
en    2497602
ru    1568927
ar    1378222
es    1360396
fr     419150
       ...   
te          2
my          1
he          1
lo          1
or          1
Name: count, Length: 71, dtype: int64

In [33]:
# Check formatting inconsistencies in categorical attributes like if it contains formatting inconsistencies, such as extra spaces or different capitalization.
for col in ["ioCountry", "post_language"]:

    # Use unique values to save memory
    values = pd.Series(df_clean[col].dropna().unique())

    normalized = values.str.strip().str.lower()

    # Find values with inconsistent formatting
    issues = pd.DataFrame({
        "Original": values,
        "Normalized": normalized
    })

    issues = issues[
        (issues["Original"] != issues["Normalized"]) |
        issues["Normalized"].duplicated(keep=False)
    ]

    print(f"\n--- {col} ---")
    print("Potential formatting inconsistencies:", len(issues))
    display(issues)


--- ioCountry ---
Potential formatting inconsistencies: 0


,Original,Normalized



--- post_language ---
Potential formatting inconsistencies: 0


,Original,Normalized


No formatting corrections are required. Missing post_language values will be addressed during the Missing Values Analysis.

Now we will move to outliers detection

In [34]:
# Examine the distributions of numerical attributes
numeric_cols = ["follower_count", "following_count"]

for col in numeric_cols:
    print(f"\n--- {col} ---")

    display(
        df_clean[col].describe(
            percentiles=[0.25, 0.5, 0.75, 0.90, 0.99, 0.999]
        )
    )

    print("Zero values:", (df_clean[col] == 0).sum())


--- follower_count ---


count    8.708217e+06
mean     3.032488e+06
std      4.458898e+08
min      0.000000e+00
25%      1.820000e+02
50%      7.140000e+02
75%      3.044000e+03
90%      1.315500e+04
99%      1.348050e+05
99.9%    2.209022e+06
max      9.958279e+10
Name: follower_count, dtype: float64

Zero values: 64908

--- following_count ---


count    8.708396e+06
mean     3.322193e+06
std      4.731284e+08
min      0.000000e+00
25%      1.660000e+02
50%      4.990000e+02
75%      1.686000e+03
90%      4.946000e+03
99%      2.907000e+04
99.9%    1.296130e+05
max      9.991536e+10
Name: following_count, dtype: float64

Zero values: 345921


The median is only 714.

99% of records have follower counts of approximately 134,805 or lower.

But the maximum is approximately 99.58 billion.


A follower count near 100 billion is not plausible for a social-media account.

Also, Extreme values are distorting the mean, lets investigate those values


In [35]:
# Investigate extreme numerical values

numeric_cols = ["follower_count", "following_count"]

for col in numeric_cols:
    print(f"\n--- {col} ---")

    # Count values above selected thresholds
    for threshold in [1e7, 1e8, 1e9, 1e10]:
        count = (df_clean[col] > threshold).sum()
        print(f"Values above {threshold:,.0f}: {count}")

    # Examine the highest values
    print("\nTop 10 highest values:")

    display(
        df_clean.nlargest(10, col)[
            ["ioCountry", "accountid", col]
        ]
    )


--- follower_count ---
Values above 10,000,000: 1310
Values above 100,000,000: 493
Values above 1,000,000,000: 493
Values above 10,000,000,000: 493

Top 10 highest values:


,ioCountry,accountid,follower_count
5043186,russia_1,4b262547190219226e276a7e6d342bc0cc42c9e81d,9.958279e+10
564116,ecuador,695748fd7fe2d32aff1ee129219199ce412bde3626,9.893545e+10
7324189,uae,6dd25ded1bd6188f6fca6beb438fbfa8bb1e3c12cd,9.871636e+10
2510911,iran_1,ce3438d4a69ae98420a7a3c449366da079d00df09b,9.850715e+10
7115197,uae,4be31b86260c6fdaf06554e5f808c2e0ad3ffff714,9.832753e+10
2456369,iran_1,ce3438d4a69ae98420a7a3c449366da079d00df09b,9.820524e+10
1025307,ecuador,a35a511e26bc2643145165bab63a5a192228f90138,9.793376e+10
322427,ecuador,2ef928f0ed37baa489e9dffd62f94838e174a015f3,9.791654e+10
7249933,uae,b9e82f4cfbea7c13563f35a06b50952a20ab4a8bf5,9.788533e+10
4028295,russia_1,7e363e1de5b0ed81518fb5f81e5f9eceb30c353b79,9.784619e+10



--- following_count ---
Values above 10,000,000: 518
Values above 100,000,000: 518
Values above 1,000,000,000: 518
Values above 10,000,000,000: 518

Top 10 highest values:


,ioCountry,accountid,following_count
2830385,iran_4,9f19a9ce73914b9bd92ecbae3f37e0bb0025e5ac00,9.991536e+10
7225632,uae,65ea9489070a939223eb5711a24e28e69cc44dbe54,9.986612e+10
1153746,ecuador,06ae71cd26b039f685ffa49c64780bd86f18fdff94,9.979599e+10
3266683,iran_4,85c91bfb9fe420a08d0df5e97465067d39fb72fe3f,9.975606e+10
4987812,russia_1,212f034522aa94b0065565821d583b12380cccdc96,9.964013e+10
3654588,russia_1,4a177c0b39e845b05b8b8196204aa9844ed0a578ff,9.945840e+10
34278,ecuador,40a927417ec7dd26939b28d3e7eeb83eddd908f1c3,9.942571e+10
5107577,russia_1,c52d67acaa4c0b8c8b84de7d3b08fb0a6ba3d2cf69,9.889349e+10
1040523,ecuador,9c77c90d9cde0cb6998c94d004453e17da4228fe82,9.887374e+10
8465520,uae,46b2ea37016db78768b66aace8037a931ad406413c,9.874355e+10


Both attributes contain extreme maximum values approaching 10 billion.

we will mark those values as Nan and handle them later

In [36]:
import numpy as np

# Threshold for clearly implausible values
threshold = 10_000_000_000

for col in ["follower_count", "following_count"]:

    invalid_extreme = df_clean[col] > threshold

    print(f"\n--- {col} ---")
    print("Invalid extreme values:", invalid_extreme.sum())

    # Replace implausible values with NaN
    df_clean.loc[invalid_extreme, col] = np.nan

    # Verify
    print("Values above threshold remaining:",
          (df_clean[col] > threshold).sum())

    print("Total missing values:",
          df_clean[col].isna().sum())


--- follower_count ---
Invalid extreme values: 493
Values above threshold remaining: 0
Total missing values: 5053

--- following_count ---
Invalid extreme values: 518
Values above threshold remaining: 0
Total missing values: 4899


Detect statistical outliers using IQR

The IQR method identifies values that are unusually high or low relative to the distribution of our dataset.

In [37]:
# Detect statistical outliers using the IQR method

for col in ["follower_count", "following_count"]:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers
    lower_outliers = (df_clean[col] < lower_bound).sum()
    upper_outliers = (df_clean[col] > upper_bound).sum()

    total_outliers = lower_outliers + upper_outliers

    print(f"\n--- {col} ---")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Lower outliers:", lower_outliers)
    print("Upper outliers:", upper_outliers)
    print("Total outliers:", total_outliers)
    print(
        "Percentage of valid observations:",
        round(
            total_outliers / df_clean[col].notna().sum() * 100,
            2
        ),
        "%"
    )


--- follower_count ---
Q1: 182.0
Q3: 3042.0
IQR: 2860.0
Lower bound: -4108.0
Upper bound: 7332.0
Lower outliers: 0
Upper outliers: 1431939
Total outliers: 1431939
Percentage of valid observations: 16.44 %

--- following_count ---
Q1: 166.0
Q3: 1686.0
IQR: 1520.0
Lower bound: -2114.0
Upper bound: 3966.0
Lower outliers: 0
Upper outliers: 1048823
Total outliers: 1048823
Percentage of valid observations: 12.04 %


We applied the IQR method to evaluate the distribution of follower_count and following_count.

follower_count: 1,431,939 observations (16.44%) exceed the upper boundary of 7,332.
following_count: 1,048,823 observations (12.04%) exceed the upper boundary of 3,966.

Consequently, the IQR method identifies many observations that may represent legitimate accounts.
For example, having more than 7,332 followers is not inherently abnormal or incorrect.

Therefore, the IQR boundaries are not appropriate for automatically identifying invalid values in this dataset.

Now we will move to missing values analysis

In [38]:
# Calculate missing values for each attribute
missing_counts = pd.Series({
    col: df_clean[col].isna().sum()
    for col in df_clean.columns
})

# Calculate percentages
missing_percentages = (
    missing_counts / len(df_clean) * 100
)

# Create the summary table
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages.round(4)
})

# Sort by number of missing values
missing_summary = missing_summary.sort_values(
    "Missing Count",
    ascending=False
)

# Display attributes with missing values
display(missing_summary[missing_summary["Missing Count"] > 0])

,Missing Count,Missing Percentage (%)
in_reply_to_accountid,7837851,89.9581
in_reply_to_postid,7826699,89.8301
reposted_postid,5108257,58.6295
reposted_accountid,5108257,58.6295
account_profile_description,1669612,19.1628
post_language,103727,1.1905
follower_count,5053,0.0580
following_count,4899,0.0562
is_control,1112,0.0128
post_text,3,0.0000


Investigate the 3 missing post texts

We want to determine whether these records still contain useful information for our project.

In [39]:
# Inspect records with missing post text
missing_text_records = df_clean.loc[
    df_clean["post_text"].isna(),
    [
        "ioCountry",
        "postid",
        "accountid",
        "post_time",
        "is_repost",
        "is_control",
        "post_language"
    ]
]

display(missing_text_records)

,ioCountry,postid,accountid,post_time,is_repost,is_control,post_language
124,ecuador,9058f856a49542a0285148bf6dd954223303d43633,fac52a138891284cd1b1640172e81cdcc7d0eca9c9,2010-08-18 02:58:24+00:00,False,True,sv
38141,ecuador,0b6aed7bf0ce7865adafa019ad08ecbe710414081c,0b4dc786afdc31053e0d9a0ca6799cee30b4487a96,2011-12-27 01:43:08+00:00,False,True,sv
38213,ecuador,f6b4b58b90bb7343b58e56f6f838097884294ae28a,0b4dc786afdc31053e0d9a0ca6799cee30b4487a96,2011-12-27 15:31:26+00:00,False,True,sv


We will keep them, since even without the text we can still use them to calculate the number of posts per acc, posting frequency etc

For account_profile_description, its normal to be empty, we can have accounts without a description

We will invesetigate the missing values in post_language
We identified 103,727 records (1.19%) with missing post_language values.
The value und (undetermined language) is distinguished from NaN, which represents unavailable language information.
lets see if those records have a post_text and if they belong to a particular ioCountry group

In [28]:
# Identify missing and undefined language values
lang = df_clean["post_language"]

# Actual missing values
missing_na = lang.isna()

# Undefined or empty language labels
undefined_lang = lang.str.fullmatch(
    r"(?i)\s*(?:und|unknown|nan|null)?\s*"
).fillna(False)

# Combine both categories
missing_lang = missing_na | undefined_lang

# Summary
print("Actual missing languages (NaN):", missing_na.sum())
print("Undefined language labels:", undefined_lang.sum())
print("Total missing/undefined:", missing_lang.sum())

print(
    "Missing/undefined language but post text available:",
    df_clean.loc[missing_lang, "post_text"].notna().sum()
)

print(
    "Both language and post text missing:",
    df_clean.loc[missing_lang, "post_text"].isna().sum()
)

# Calculate missing/undefined percentages by country
total_by_country = df_clean["ioCountry"].value_counts()

missing_by_country = (
    df_clean.loc[missing_lang, "ioCountry"]
    .value_counts()
)

country_summary = pd.DataFrame({
    "Total Posts": total_by_country,
    "Missing/Undefined Language": missing_by_country
}).fillna(0)

country_summary["Missing (%)"] = (
    country_summary["Missing/Undefined Language"]
    / country_summary["Total Posts"] * 100
).round(2)

display(country_summary)

# Inspect some affected posts
display(
    df_clean.loc[
        missing_lang,
        ["ioCountry", "post_language", "post_text", "is_control"]
    ].head(5)
)

Actual missing languages (NaN): 103727
Undefined language labels: 373032
Total missing/undefined: 476759
Missing/undefined language but post text available: 476759
Both language and post text missing: 0


,Total Posts,Missing/Undefined Language,Missing (%)
ioCountry,,,
ecuador,1470529,100022,6.80
iran_1,1322789,58210,4.40
iran_4,561585,30499,5.43
russia_1,3650082,194756,5.34
uae,1707792,93272,5.46


,ioCountry,post_language,post_text,is_control
0,ecuador,NaN,Vamos a cambiar la forma de hacer comunicación...,False
1,ecuador,NaN,"Damos servicio de manejo de prensa, elaboració...",False
2,ecuador,NaN,@db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302 Bu...,False
3,ecuador,NaN,"Mario, debemos denuciar que en los poderes pub...",False
4,ecuador,NaN,Tambien debemos estar claro que la oposición e...,False


for those values, we will try to use a language-detection API to identify the language based on the post_text

we will first test it one some records to see the accuracy, and then we can apply it on the whole data

we will do that by taking a representative sample, from our dataset, delete the labels, and then check if it will be able to detect its language

In [7]:
from fast_langdetect import detect

result = detect(
    "Hola, ¿cómo estás?",
    model="lite",
    k=1
)

print(result)

[{'lang': 'es', 'score': 0.9931496381759644}]


In [12]:
import pandas as pd
import numpy as np

SAMPLE_SIZE = 500
CHUNK_SIZE = 50000

rng = np.random.default_rng(42)

sample = pd.DataFrame()
total_eligible = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["post_text", "post_language"],
    dtype="string",
    chunksize=CHUNK_SIZE
):
    # Normalize language labels
    chunk["post_language"] = (
        chunk["post_language"]
        .str.strip()
        .str.lower()
    )

    # Keep records with known language and available text
    valid = (
        chunk["post_language"].notna()
        & ~chunk["post_language"].isin(
            ["und", "unknown", ""]
        )
        & chunk["post_text"].notna()
        & chunk["post_text"].str.strip().ne("")
    ).fillna(False)

    eligible = chunk.loc[
        valid, ["post_text", "post_language"]
    ].copy()

    total_eligible += len(eligible)

    if eligible.empty:
        continue

    # Assign random priorities to posts
    eligible["_priority"] = rng.random(len(eligible))

    # Keep only the best 500 random candidates
    candidates = eligible.nsmallest(
        SAMPLE_SIZE, "_priority"
    )

    sample = (
        pd.concat([sample, candidates], ignore_index=True)
        .nsmallest(SAMPLE_SIZE, "_priority")
    )

# Final randomized sample
test_df = (
    sample
    .drop(columns="_priority")
    .rename(columns={"post_language": "true_language"})
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Total eligible posts:", total_eligible)
print("Total sampled posts:", len(test_df))
print("Languages represented:", test_df["true_language"].nunique())

display(test_df["true_language"].value_counts())
display(test_df.head())

Total eligible posts: 8237508
Total sampled posts: 500
Languages represented: 22


true_language
en    166
ru    100
es     78
ar     75
fa     22
fr     16
pt     11
tr      7
in      4
de      3
tl      3
ht      3
uk      3
ro      1
ca      1
lv      1
ja      1
cy      1
fi      1
et      1
sr      1
bg      1
Name: count, dtype: int64[pyarrow]

,post_text,true_language
0,Мун Чжэ Ин: Ким Чен Ын не доверяет обещаниям С...,ru
1,Romney hints at 3rd campaign in speech to nati...,en
2,I have a rising follower graph! 1 more followe...,en
3,@d43487e0e2181c76e94ad30e8da1537b181f9c9030 fi...,pt
4,GOOOOOO bueno AUTOGOOOOOL Colvey anota el cua...,es


In [14]:
# Predict languages locally
test_df["predicted_language"] = test_df["post_text"].apply(
    lambda text: detect(
        str(text),
        model="lite",
        k=1
    )[0]["lang"]
)

# Compare predictions with original labels
test_df["correct"] = (
    test_df["true_language"].str.lower()
    == test_df["predicted_language"].str.lower()
)

print(f"Total posts: {len(test_df)}")
print(f"Correct predictions: {test_df['correct'].sum()}")
print(f"Accuracy: {test_df['correct'].mean() * 100:.2f}%")

# Accuracy by language
display(
    test_df.groupby("true_language")["correct"]
    .agg(
        total="count",
        correct="sum",
        accuracy="mean"
    )
    .round(3)
)

# Inspect incorrect predictions
display(
    test_df.loc[
        ~test_df["correct"],
        ["true_language", "predicted_language", "post_text"]
    ].head(20)
)

Total posts: 500
Correct predictions: 449
Accuracy: 89.80%


,total,correct,accuracy
true_language,,,
ar,75,73,0.973
bg,1,0,0.0
ca,1,1,1.0
cy,1,0,0.0
de,3,2,0.667
en,166,156,0.94
es,78,68,0.872
et,1,0,0.0
fa,22,21,0.955


,true_language,predicted_language,post_text
9,in,en,cinco #MTVHottest Katy Perry
14,ru,uk,НЕТ ФУТБОЛА БЕЗ ФРЕНЧ ДОГА https://d5d7b6a8df1...
18,ro,en,gsdyb #MTVHottest Iggy Azalea Ellie Goulding R...
26,en,ja,@50751804b8e80b5f11fab4bad0e0f56a312af2d12f @5...
36,en,ca,@efcfec601b9fecbaedc423b4ed0f37141ceb809e8c @5...
37,es,pl,@bef0e7981715abed0539a1eb8eedfdbf59651779f2 @a...
41,lv,en,Maldives 🇲🇻 https://12b1455387d517482d480e4078...
51,en,fr,@bc7e137c3bf276edcae4697224ee604f381229b7dc in...
57,en,es,I love mirror :)
68,es,pt,@2f0636e084dc126973d1eba6642dfb4b2aab1c3c5d JA...


some langauge codes in our database are old, there are new codes for them
   "in": "id",  # Indonesian
    "iw": "he",  # Hebrew
    "ji": "yi"   # Yiddish

In [15]:
# Normalize equivalent language codes
language_mapping = {
    "in": "id",  # Indonesian
    "iw": "he",  # Hebrew
    "ji": "yi"   # Yiddish
}

true_labels = (
    test_df["true_language"]
    .str.lower()
    .replace(language_mapping)
)

predicted_labels = (
    test_df["predicted_language"]
    .str.lower()
    .replace(language_mapping)
)

# Recalculate accuracy
test_df["correct_normalized"] = (
    true_labels == predicted_labels
)

print(
    "Accuracy after normalization:",
    f"{test_df['correct_normalized'].mean() * 100:.2f}%"
)

# Examine remaining errors
errors = test_df.loc[
    ~test_df["correct_normalized"],
    ["true_language", "predicted_language", "post_text"]
]

print("Remaining disagreements:", len(errors))
display(errors.head(25))

Accuracy after normalization: 90.20%
Remaining disagreements: 49


,true_language,predicted_language,post_text
9,in,en,cinco #MTVHottest Katy Perry
14,ru,uk,НЕТ ФУТБОЛА БЕЗ ФРЕНЧ ДОГА https://d5d7b6a8df1...
18,ro,en,gsdyb #MTVHottest Iggy Azalea Ellie Goulding R...
26,en,ja,@50751804b8e80b5f11fab4bad0e0f56a312af2d12f @5...
36,en,ca,@efcfec601b9fecbaedc423b4ed0f37141ceb809e8c @5...
37,es,pl,@bef0e7981715abed0539a1eb8eedfdbf59651779f2 @a...
41,lv,en,Maldives 🇲🇻 https://12b1455387d517482d480e4078...
51,en,fr,@bc7e137c3bf276edcae4697224ee604f381229b7dc in...
57,en,es,I love mirror :)
68,es,pt,@2f0636e084dc126973d1eba6642dfb4b2aab1c3c5d JA...


In [33]:
import re
import pandas as pd

def clean_for_language_detection(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove usernames and anonymized mentions
    text = re.sub(r"(?<!\w)@\S+", " ", text)

    # Remove hashtag symbols but keep their text
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [34]:
def predict_cleaned_language(text):
    if not any(char.isalpha() for char in text):
        return "unknown"

    return detect(text, model="lite", k=1)[0]["lang"]

test_df["cleaned_prediction"] = (
    test_df["clean_text"].apply(predict_cleaned_language)
)

In [35]:
language_mapping = {
    "in": "id",
    "iw": "he",
    "ji": "yi"
}

true = test_df["true_language"].replace(language_mapping)
original = test_df["predicted_language"].replace(language_mapping)
cleaned = test_df["cleaned_prediction"].replace(language_mapping)

original_correct = true == original
cleaned_correct = true == cleaned

print(f"Original accuracy: {original_correct.mean() * 100:.2f}%")
print(f"After cleaning:    {cleaned_correct.mean() * 100:.2f}%")

print("Errors corrected:", ((~original_correct) & cleaned_correct).sum())
print("New errors:", (original_correct & (~cleaned_correct)).sum())

# Examine predictions that changed
display(
    test_df.loc[
        original != cleaned,
        [
            "true_language",
            "predicted_language",
            "cleaned_prediction",
            "post_text",
            "clean_text"
        ]
    ].head(20)
)

Original accuracy: 90.20%
After cleaning:    93.20%
Errors corrected: 16
New errors: 1


,true_language,predicted_language,cleaned_prediction,post_text,clean_text
26,en,ja,en,@50751804b8e80b5f11fab4bad0e0f56a312af2d12f @5...,what they will never say you about Merle Hagga...
36,en,ca,en,@efcfec601b9fecbaedc423b4ed0f37141ceb809e8c @5...,Follow = back
37,es,pl,es,@bef0e7981715abed0539a1eb8eedfdbf59651779f2 @a...,Que bueno presi que se impulse la #ExcelenciaP...
51,en,fr,en,@bc7e137c3bf276edcae4697224ee604f381229b7dc in...,inv ReeseeY
68,es,pt,es,@2f0636e084dc126973d1eba6642dfb4b2aab1c3c5d JA...,"JAJAJAJAJA sjdfnjsdnfjsd amo el indio, ah ksdm..."
119,fr,eu,fr,@2b45c69dab66b91cdeb7a11cbb450954eea6b0985e @3...,comment François Hollande a redoré l'image de ...
130,en,vi,en,@38073c11f0d371557ce99fa704b4115d02240340be gl...,glad to see a link here...
157,ru,en,ru,@7fae0977a63aa9036cf0d099dced71820d3944e64c @8...,не назад ))) а всем вперед!!!
169,in,la,en,"HHHHHHHHHHHH RT ""@ebbc1acc9fc925573960cc7a53f7...","HHHHHHHHHHHH RT "" S1 و"""
191,pt,es,pt,SIGO\n TODOS\n ...,SIGO TODOS DE VOLTA #RT


Notice the row of id 278, the true language in the psot text is english, but in the database its tl
That means the original dataset's labels may sometimes be inaccurate or may describe a language not obvious from the visible text.

In [36]:
def detect_with_confidence(text):
    if not text or not any(c.isalpha() for c in str(text)):
        return "unknown", 0.0

    result = detect(str(text), model="lite", k=1)[0]
    return result["lang"], result["score"]

results = test_df["clean_text"].apply(detect_with_confidence)

test_df["confidence_language"] = results.apply(lambda x: x[0])
test_df["confidence_score"] = results.apply(lambda x: x[1])

true = test_df["true_language"].replace({
    "in": "id", "iw": "he", "ji": "yi"
})

predicted = test_df["confidence_language"].replace({
    "in": "id", "iw": "he", "ji": "yi"
})

for threshold in [0.3,0.4,0.45,0.5, 0.7, 0.8, 0.9, 0.95]:
    accepted = test_df["confidence_score"] >= threshold

    accuracy = (
        true[accepted] == predicted[accepted]
    ).mean() * 100 if accepted.any() else float("nan")

    print(
        f"Threshold {threshold:.2f} | "
        f"Accuracy: {accuracy:.2f}% | "
        f"Coverage: {accepted.mean() * 100:.1f}%"
    )

Threshold 0.30 | Accuracy: 94.03% | Coverage: 97.2%
Threshold 0.40 | Accuracy: 95.74% | Coverage: 93.8%
Threshold 0.45 | Accuracy: 96.72% | Coverage: 91.6%
Threshold 0.50 | Accuracy: 97.30% | Coverage: 88.8%
Threshold 0.70 | Accuracy: 98.69% | Coverage: 76.2%
Threshold 0.80 | Accuracy: 98.75% | Coverage: 64.2%
Threshold 0.90 | Accuracy: 99.25% | Coverage: 53.0%
Threshold 0.95 | Accuracy: 99.48% | Coverage: 38.2%


In [37]:
import pandas as pd
import gzip
import re
from pathlib import Path
from fast_langdetect import detect

# 1. File paths
DATA_PATH = Path("data_processed/df_clean_final.csv.gz")

OUTPUT_PATH = DATA_PATH.with_name(
    "df_clean_language_imputed.csv.gz"
)

TEMP_PATH = OUTPUT_PATH.with_name(
    OUTPUT_PATH.name + ".partial"
)

CHUNK_SIZE = 20000
CONFIDENCE_THRESHOLD = 0.50


# 2. Text preprocessing used in the experiment
def clean_for_language_detection(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove usernames and anonymized mentions
    text = re.sub(r"(?<!\w)@\S+", " ", text)

    # Normalize whitespace
    return re.sub(r"\s+", " ", text).strip()


# 3. Counters
total_missing = 0
total_imputed = 0
total_rejected = 0
total_no_text = 0


# 4. Process dataset in chunks
with gzip.open(
    TEMP_PATH,
    "wt",
    encoding="utf-8",
    newline=""
) as output:

    for chunk_num, chunk in enumerate(
        pd.read_csv(
            DATA_PATH,
            dtype="string",
            chunksize=CHUNK_SIZE
        )
    ):

        language = (
            chunk["post_language"]
            .str.strip()
            .str.lower()
        )

        # Identify missing and undefined languages
        missing = (
            language.isna()
            | language.isin(
                ["", "und", "unknown", "nan", "null"]
            )
        ).fillna(False)

        total_missing += int(missing.sum())

        # Track imputed records
        chunk["language_imputed"] = False
        chunk["language_confidence"] = pd.Series(
            pd.NA,
            index=chunk.index,
            dtype="Float64"
        )

        for idx in chunk.index[missing]:

            text = clean_for_language_detection(
                chunk.at[idx, "post_text"]
            )

            if not any(c.isalpha() for c in text):
                total_no_text += 1
                continue

            result = detect(
                text,
                model="lite",
                k=1
            )[0]

            predicted_language = result["lang"]
            confidence = float(result["score"])

            chunk.at[idx, "language_confidence"] = confidence

            if (
                confidence >= CONFIDENCE_THRESHOLD
                and predicted_language != "unknown"
            ):
                chunk.at[idx, "post_language"] = predicted_language
                chunk.at[idx, "language_imputed"] = True
                total_imputed += 1
            else:
                total_rejected += 1

        chunk.to_csv(
            output,
            index=False,
            header=(chunk_num == 0)
        )

        print(
            f"Chunk {chunk_num + 1} | "
            f"Missing found: {total_missing:,} | "
            f"Imputed: {total_imputed:,}",
            end="\r"
        )

# Save completed file
TEMP_PATH.replace(OUTPUT_PATH)


# 5. Summary
print("\n\nLanguage Imputation Summary")
print("----------------------------------")
print(f"Missing before:  {total_missing:,}")
print(f"Imputed:         {total_imputed:,}")
print(f"Low confidence:  {total_rejected:,}")
print(f"No usable text:  {total_no_text:,}")
print(f"Still missing:   {total_missing - total_imputed:,}")

if total_missing > 0:
    print(
        f"Recovery rate:   "
        f"{total_imputed / total_missing * 100:.2f}%"
    )

print("\nSaved to:", OUTPUT_PATH)

Chunk 436 | Missing found: 476,759 | Imputed: 280,543

Language Imputation Summary
----------------------------------
Missing before:  476,759
Imputed:         280,543
Low confidence:  170,532
No usable text:  25,684
Still missing:   196,216
Recovery rate:   58.84%

Saved to: data_processed\df_clean_language_imputed.csv.gz


In [40]:
import pandas as pd
import numpy as np
import re

# 1. Text cleaning function
def clean_review_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove usernames and mentions
    text = re.sub(r"(?<!\w)@\S+", " ", text)

    # Remove hashtags
    text = re.sub(r"(?<!\w)#\S+", " ", text)

    # Normalize whitespace
    return re.sub(r"\s+", " ", text).strip()


# 2. Randomly select 30 imputed posts
rng = np.random.default_rng(42)
review_sample = pd.DataFrame()

for chunk in pd.read_csv(
    OUTPUT_PATH,
    usecols=[
        "ioCountry",
        "post_text",
        "post_language",
        "language_imputed",
        "language_confidence"
    ],
    dtype="string",
    chunksize=100000
):
    imputed = (
        chunk["language_imputed"]
        .str.lower()
        .eq("true")
        .fillna(False)
    )

    candidates = chunk.loc[imputed].copy()

    if candidates.empty:
        continue

    candidates["_priority"] = rng.random(len(candidates))

    review_sample = (
        pd.concat(
            [review_sample, candidates.nsmallest(30, "_priority")],
            ignore_index=True
        )
        .nsmallest(30, "_priority")
    )


# 3. Clean the selected posts
review_sample["clean_text"] = (
    review_sample["post_text"].apply(clean_review_text)
)

# Count remaining alphabetic characters
review_sample["letter_count"] = (
    review_sample["clean_text"]
    .apply(lambda text: sum(c.isalpha() for c in text))
)

# 4. Display results
display(
    review_sample[
        [
            "ioCountry",
            "post_text",
            "clean_text",
            "letter_count",
            "post_language",
            "language_confidence"
        ]
    ].reset_index(drop=True)
)

print(
    "Posts with no alphabetic text:",
    (review_sample["letter_count"] == 0).sum()
)

,ioCountry,post_text,clean_text,letter_count,post_language,language_confidence
0,ecuador,"""@8c19824973fccd410341b89f19555f857a8b1f76fe: ...",""" TODOS queremos retroceder el tiempo alguna v...",61,es,0.954268217086792
1,iran_1,.@a434be2c58dd0622c9cb1d6f644f7d83bf60415cbb 2...,. 2 4,0,en,0.7385226488113403
2,russia_1,#Красавицы https://21498519e21382421b810ace217...,,0,ru,0.8337535262107849
3,ecuador,O https://4daa964fb274e5a5cb0d7e859a87afef72f6...,O,1,en,0.5795838832855225
4,russia_1,#Стабильность https://5d7b1fa3b9ff4bbdeb2bb688...,,0,ru,0.9269818663597107
5,russia_1,"@060550842afd65368fa910617e8e5c3cf522726118 ""Д...","""Динамо"" обыграло ""Анжи"" в Махачкале",28,ru,0.9133388996124268
6,ecuador,https://898657c0c4503ba6fe4df770d0eea15ebc503b...,,0,en,0.785908043384552
7,russia_1,https://c90c13162422108e0898a2a67c9be792844c87...,,0,en,0.6936748027801514
8,russia_1,#muslimwomensday @afb9114c251602153aaa4a1ad4fd...,,0,en,0.6950072050094604
9,uae,#نحو_الريو,,0,ar,0.9539409875869751


Posts with no alphabetic text: 19


### Language Detection Experiment — Conclusion

We investigated whether missing and undefined `post_language` values could be recovered using fastText, a free language-detection model.

We applied the model to missing and undefined language records using a confidence threshold of 0.50.

We inspected 30 randomly selected imputed posts and identified several limitations:

- Some predictions were based on URLs, usernames, hashtags, or very short text.
- Certain posts contained insufficient linguistic information for reliable language detection.
- Some posts received relatively high confidence scores despite containing limited meaningful text.

**Final Decision:** We decided not to use the experimentally imputed languages in our final dataset to avoid introducing potentially incorrect information into subsequent analysis.


We have previously identified:

- 5,053 missing follower_count values.
- 4,899 missing following_count values.

Since the dataset contains multiple posts per account, some missing values may be recoverable using other observations from the same account. We will check whether the accounts affected by missing values have other observations containing valid counts.

In [44]:
# Check whether missing numerical values can be recovered
# from other observations of the same account

for col in ["follower_count", "following_count"]:

    # Records with missing values
    missing_rows = df_clean.loc[
        df_clean[col].isna(),
        ["ioCountry", "accountid"]
    ].copy()

    # Find accounts involved in those records
    affected_ids = missing_rows["accountid"].dropna().unique()

    # Find the same accounts with available values
    available_accounts = (
        df_clean.loc[
            df_clean["accountid"].isin(affected_ids)
            & df_clean[col].notna(),
            ["ioCountry", "accountid"]
        ]
        .drop_duplicates()
    )

    # Match missing records to accounts with valid observations
    comparison = missing_rows.merge(
        available_accounts.assign(has_valid_value=True),
        on=["ioCountry", "accountid"],
        how="left"
    )

    recoverable = comparison["has_valid_value"].eq(True).sum()

    print(f"\n--- {col} ---")
    print("Total missing records:", len(missing_rows))
    print("Records with another valid account observation:",
          recoverable)
    print("Records without another valid account observation:",
          len(missing_rows) - recoverable)


--- follower_count ---
Total missing records: 5053
Records with another valid account observation: 5049
Records without another valid account observation: 4

--- following_count ---
Total missing records: 4899
Records with another valid account observation: 4896
Records without another valid account observation: 3


Most missing follower and following counts have valid observations from the same account.

We will estimate missing values using the closest observation in time from the same `(ioCountry, accountid)`.

Records without valid observations will remain missing. 


In [45]:
for col in ["follower_count", "following_count"]:

    # Records with missing values
    missing = df_clean.loc[
        df_clean[col].isna(),
        ["ioCountry", "accountid", "post_time"]
    ].dropna().copy()

    missing["row_index"] = missing.index

    # Valid observations from the affected accounts
    valid = df_clean.loc[
        df_clean[col].notna()
        & df_clean["accountid"].isin(missing["accountid"]),
        ["ioCountry", "accountid", "post_time", col]
    ].dropna()

    # Find the closest observation in time
    matched = pd.merge_asof(
        missing.sort_values("post_time"),
        valid.sort_values("post_time"),
        on="post_time",
        by=["ioCountry", "accountid"],
        direction="nearest"
    )

    # Replace missing values
    found = matched[col].notna()

    df_clean.loc[
        matched.loc[found, "row_index"], col
    ] = matched.loc[found, col].to_numpy()

    print(f"{col}: {found.sum()} values filled")
    print(f"Remaining missing: {df_clean[col].isna().sum()}\n")

follower_count: 5049 values filled
Remaining missing: 4

following_count: 4896 values filled
Remaining missing: 3



### Handling Missing `is_control` Values

We will check whether missing labels can be recovered from other posts belonging to the same `(ioCountry, accountid)`.

A missing value can be replaced only if all available labels for that account agree. Conflicting or unavailable labels will remain missing.

In [46]:
# Records with missing is_control
missing = df_clean.loc[
    df_clean["is_control"].isna(),
    ["ioCountry", "accountid"]
]

# Find existing labels for affected accounts
known = df_clean.loc[
    df_clean["accountid"].isin(missing["accountid"].dropna())
    & df_clean["is_control"].notna(),
    ["ioCountry", "accountid", "is_control"]
]

# Check whether labels agree within each account and country
account_labels = (
    known.groupby(["ioCountry", "accountid"])["is_control"]
    .agg(["nunique", "first"])
    .reset_index()
)

# Match missing records to available account labels
comparison = missing.merge(
    account_labels,
    on=["ioCountry", "accountid"],
    how="left"
)

print("Total missing:", len(comparison))
print("Recoverable (consistent label):", (comparison["nunique"] == 1).sum())
print("Conflicting labels:", (comparison["nunique"] > 1).sum())
print("No available label:", comparison["nunique"].isna().sum())

Total missing: 1112
Recoverable (consistent label): 1110
Conflicting labels: 2
No available label: 0


In [47]:
# Keep accounts with one consistent is_control label
consistent = account_labels.loc[
    account_labels["nunique"] == 1,
    ["ioCountry", "accountid", "first"]
]

# Find missing records while preserving their original indices
missing = df_clean.loc[
    df_clean["is_control"].isna(),
    ["ioCountry", "accountid"]
].copy()

missing["row_index"] = missing.index

# Match missing records with consistent account labels
matched = missing.merge(
    consistent,
    on=["ioCountry", "accountid"],
    how="left",
    validate="many_to_one"
)

# Replace only recoverable values
fillable = matched["first"].notna()

df_clean.loc[
    matched.loc[fillable, "row_index"],
    "is_control"
] = matched.loc[fillable, "first"].to_numpy()

print("Values replaced:", fillable.sum())
print("Remaining missing:", df_clean["is_control"].isna().sum())

Values replaced: 1110
Remaining missing: 2


### Missing Post Text and Account Description

Post text and account profile descriptions are treated as optional attributes.

Missing values are preserved because the records still contain useful information for our analysis. No imputation or deletion is required.

Missing Repost and Reply Information

Missing values in repost and reply IDs were previously examined during the data consistency assessment.

- Repost IDs: reposted_postid and reposted_accountid each have 5,108,265 missing values. All belong to posts where is_repost = False, so these missing values are expected.

- Reply IDs: in_reply_to_postid has 7,826,711 missing values, while in_reply_to_accountid has 7,837,863. In 7,763,255 records, both IDs are missing, likely because those posts are not replies. Some records have only one of the two IDs, indicating incomplete metadata.

Some records have an available in_reply_to_postid but a missing in_reply_to_accountid.
We can attempt to recover the missing account ID by searching for the referenced post in the dataset and check its account_id.

In [48]:
# 1. Find records with a reply post ID but missing reply account ID
missing = df_clean.loc[
    df_clean["in_reply_to_accountid"].isna()
    & df_clean["in_reply_to_postid"].notna(),
    ["ioCountry", "in_reply_to_postid"]
].copy()

missing["row_index"] = missing.index

# 2. Find referenced posts in our dataset
candidates = df_clean.loc[
    df_clean["postid"].isin(missing["in_reply_to_postid"]),
    ["ioCountry", "postid", "accountid"]
]

# 3. Count distinct known authors for each post within each country
lookup = (
    candidates.groupby(["ioCountry", "postid"])
    .agg(
        n_accounts=("accountid", "nunique"),
        recovered_accountid=("accountid", "first")
    )
    .reset_index()
)

# 4. Match missing records with referenced posts
matched = missing.merge(
    lookup,
    left_on=["ioCountry", "in_reply_to_postid"],
    right_on=["ioCountry", "postid"],
    how="left",
    validate="many_to_one"
)

# 5. Identify why some values cannot be recovered
found_anywhere = matched["in_reply_to_postid"].isin(
    candidates["postid"]
)

recoverable = matched["n_accounts"].eq(1)
not_found = ~found_anywhere
different_country = found_anywhere & matched["n_accounts"].isna()
no_known_author = matched["n_accounts"].eq(0)
conflicting_authors = matched["n_accounts"].gt(1)

# 6. Show results
print("Total missing reply account IDs:", len(matched))
print("Recoverable:", recoverable.sum())

print("\nUnrecoverable reasons:")
print("Post ID not found anywhere:", not_found.sum())
print("Post ID found only in another country:", different_country.sum())
print("Post found but author ID missing:", no_known_author.sum())
print("Post ID associated with multiple accounts:", conflicting_authors.sum())

print("\nTotal unrecoverable:", (~recoverable).sum())

Total missing reply account IDs: 74608
Recoverable: 4020

Unrecoverable reasons:
Post ID not found anywhere: 70584
Post ID found only in another country: 4
Post found but author ID missing: 0
Post ID associated with multiple accounts: 0

Total unrecoverable: 70588


### Results: Missing Reply IDs

- **Missing `in_reply_to_accountid` (74,608 records):** We searched for the referenced posts using `in_reply_to_postid` and `ioCountry`. We identified 4,020 potentially recoverable account IDs, while 70,588 could not be reliably recovered.

- **Missing `in_reply_to_postid` (63,456 records):** Although `in_reply_to_accountid` is available, an account may have published multiple posts. Therefore, we cannot determine which specific post was replied to.

**Decision:** Recover the 4,020 unambiguous account IDs and preserve the remaining missing values.

In [49]:
# Select records with one uniquely identified account
recoverable = (
    matched["n_accounts"].eq(1)
    & matched["recovered_accountid"].notna()
)

# Get original dataframe indices and recovered account IDs
row_indices = matched.loc[recoverable, "row_index"].to_numpy()
account_ids = matched.loc[recoverable, "recovered_accountid"].to_numpy()

# Fill missing reply account IDs
df_clean.loc[row_indices, "in_reply_to_accountid"] = account_ids

# Verify results
print("Values recovered:", len(row_indices))
print(
    "Remaining missing reply account IDs:",
    df_clean["in_reply_to_accountid"].isna().sum()
)

Values recovered: 4020
Remaining missing reply account IDs: 7833831


In [50]:
columns = [
    "follower_count",
    "following_count",
    "is_control",
    "post_text",
    "post_language",
    "account_profile_description",
    "reposted_postid",
    "reposted_accountid",
    "in_reply_to_postid",
    "in_reply_to_accountid"
]

print(df_clean[columns].isna().sum())

follower_count                       4
following_count                      3
is_control                           2
post_text                            3
post_language                   103727
account_profile_description    1669612
reposted_postid                5108257
reposted_accountid             5108257
in_reply_to_postid             7826699
in_reply_to_accountid          7833831
dtype: int64


In [51]:
from pathlib import Path

output_path = Path("data_processed/df_clean_final.csv.gz")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_path,
    index=False,
    compression={"method": "gzip", "compresslevel": 1},
    chunksize=25_000
)

print("Saved successfully:", output_path.exists())
print("File size (MB):", round(output_path.stat().st_size / 1024**2, 2))

Saved successfully: True
File size (MB): 1428.97
